In [ ]:
from datetime import date
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

df = catalog.load('raw/openaire/researchproduct_dev#parquet')

In [ ]:
def _pick_load_dt(df: pd.DataFrame):
    # Si hay una sola fecha en el batch, usala; si hay varias, quedate con la más reciente;
    # si no hay, hoy.
    if 'load_datetime' not in df.columns or df['_load_datetime'].isna().all():
        return date.today()
    vals = df['_load_datetime'].dropna()
    if vals.nunique() == 1:
        return vals.iloc[0]
    return pd.to_datetime(vals).max().date()

In [ ]:
df_research_subjects = df.loc[:,['id','subjects']]
df_research_subjects.dropna(inplace=True)

In [ ]:
df_research_subjects

In [ ]:
df_research_subjects = df_research_subjects.explode('subjects').reset_index(drop=True)

In [ ]:
df_research_subjects

In [ ]:
df_subjects = pd.json_normalize(df_research_subjects['subjects'])
df_research_subjects = pd.concat([df_research_subjects['id'], df_subjects],axis=1)

In [ ]:
df_research_subjects

## Paso 1: Convierto tipos y selecciono columnas con cardinalidad 1 con respecto a cada research product
+ info en https://graph.openaire.eu/docs/data-model/entities/research-product

In [ ]:
def openaire_land_researchproduct_subjects(df: pd.DataFrame)-> pd.DataFrame:

    load_dt = _pick_load_dt(df)

    df_research_subjects = df.loc[:,['id','subjects']]
    df_research_subjects.dropna(inplace=True)

    df_research_subjects = df_research_subjects.explode('subjects').reset_index(drop=True)

    df_subjects = pd.json_normalize(df_research_subjects['subjects'])
    df_research_subjects = pd.concat([df_research_subjects['id'], df_subjects],axis=1)

    df_research_subjects['_load_datetime'] = load_dt

    return df_research_subjects


In [ ]:
df_research_subjects = openaire_land_researchproduct_subjects(df)

In [ ]:
df_research_subjects